<img src="2026-09-23-13-08-16.png" width=300>

In [1]:
import os
import getpass
from dotenv import load_dotenv

load_dotenv()

def _set_env(var: str):
    if not os.environ.get(var):
        os.environ[var] = getpass.getpass(f"var: ")

_set_env("ANTHROPIC_API_KEY")

In [2]:
from anthropic import Anthropic

client = Anthropic(api_key=os.getenv("ANTHROPIC_API_KEY"))

In [6]:
MODEL = "claude-sonnet-5"
MAX_TOKENS = 4096
# test the llm can answer questions

def llm_call(user_input: str) -> str:
    response = client.messages.create(
        model=MODEL,
        max_tokens=MAX_TOKENS,
        messages=[
            {"role": "user", "content": user_input}
        ]
    )
    return response.content[0].text

llm_call("Hi!")

"Hi! How's it going? What can I help you with today?"

In [8]:
def llm_chat(user_input: str, messages: list = []) -> str:
    message = {"role": "user", "content": user_input}
    messages.append(message)
    response = client.messages.create(
        model=MODEL,
        max_tokens=MAX_TOKENS,
        messages=messages
    )
    
    return messages,response.content[-1].text

messages, input_1 = llm_chat("Hi my name is Lucas")
input_2 = "Hi waht's my name?"
llm_chat(input_2, messages)

([{'role': 'user', 'content': 'Hi my name is Lucas'},
  {'role': 'user', 'content': "Hi waht's my name?"}],
 'Your name is Lucas! You just told me that. 😊 How can I help you today?')

<img src="2026-09-23-13-16-38.png" width=400>

In [ ]:
SYSTEM_MSG = "You use web search tool to search the web and provide answers."

web_search_tool = {"type": "web_search_20260318",
                   "name": "web_search"}

tools = [web_search_tool]

def llm_chat_with_tool(user_input: str, messages: list = [], tools: list = []) -> str:
    message = {"role": "user", "content": user_input}
    messages.append(message)
    response = client.messages.create(
        model=MODEL,
        max_tokens=MAX_TOKENS,
        system=SYSTEM_MSG,
        messages=messages,
        tools=tools
    )
        
    return messages,response

msgs, response = llm_chat_with_tool("search for cool things to do in London.", tools=tools)

In [23]:
print(response.stop_reason)
"".join(b.text for b in response.content if b.type == "text")

".\n\n**Free Annual Events**\nIf you're visiting at the right time, London also puts on some great free spectac"

# Agent Loop

<img src="2026-09-23-13-28-44.png" width=500>


In [31]:
from anthropic import beta_tool
import json

SYSTEM_MSG = """
You are a personal research agent. You can search the web and
organize your findings as markdown files in the user's workspace.
when asked to research X, search the web, save a markdown file for it
with brief + sources. Keep filenames lowercase-hyphenated.
"""

@beta_tool
def write_file(file_path: str, content: str) -> str:
    """Writes to file"""
    with open(file_path, "w") as f:
        f.write(content)

write_file_tool_schema = write_file.to_dict()

web_search_tool = {"type": "web_search_20260318",
                   "name": "web_search"}

tools = [web_search_tool]

available_functions = {"web_search": web_search_tool, "write_file": write_file}

def agent_loop(user_query: str, max_iterations: int = 10) -> str:
    messages = [{"role": "user", "content": user_query}]
    print(f"User: {user_query}\n")
    
    for i in range(max_iterations):
        response = client.messages.create(
                    model="claude-sonnet-5",
                    max_tokens=2048,
                    system=SYSTEM_MSG,
                    messages=messages,
                    tools=tools,
                )
        if response.stop_reason != "tool_use":
            text = "".join(b.text for b in response.content if b.type == "text")
            print(f"Agent (final): {text}")
            return text
        
        messages.append({"role": "assistant", "content": response.content})
        tool_results = []
        for block in response.content:
            if block.type == "tool_use":
                print(f"  [{i+1}] {block.name}({json.dumps(block.input)[:120]})")
                try:
                    result = available_functions[block.name](**block.input)
                except Exception as e:
                    result = f"Error: {e}"
                tool_results.append({
                    "type": "tool_result",
                    "tool_use_id": block.id,
                    "content": str(result),
                })
        messages.append({"role": "user", "content": tool_results})

    return "(max iterations reached)"

In [32]:
response = agent_loop("Search the model context protocol docs for the modern communication flow with the updates to the protocol")
response

User: Search the model context protocol docs for the modern communication flow with the updates to the protocol



TypeError: sequence item 0: expected str instance, NoneType found

# Let's bring in MCP Servers into the Picture!